# Trustlayer with your own model

Trustlayer adds calibrated bounds to the predictions of any model. This notebook uses a model that has nothing to do with GridFM: a ridge regression written in a few lines of numpy, which predicts the voltage magnitudes `Vm` and the line loadings of the IEEE 14-bus grid from the loads and the outages of each scenario. It goes through what you would do with your own model, using the Python API only (no config, no GridFM file layout):

1. put the predictions and true values in long tables, one row per scenario and bus or line;
2. turn them into `ChannelSet`s with `ChannelSet.from_frame`, and check them with `check_inputs`;
3. fit split conformal (`NaiveIntervalCalibrator`, SCP) and kernel-weighted (`KernelIntervalCalibrator`, KCP) bounds;
4. check the coverage on test scenarios with `coverage_report`, and screen them against operating limits with `screening_report`;
5. apply the bounds to new scenarios, for which only the model's predictions exist, and get the results back as a table with `to_frame`.

The data are the contingency scenarios of the [demo](../demo/README.md) (one to five outages of lines or generators, solved with an AC power flow), read from `../demo/data`. The notebook runs in a few seconds and writes no files. The guide [Using trustlayer with your own data](https://gridfm.github.io/gridfm-trustlayer/user-guide/own-data/) (also printed by `gridfm_trustlayer guide`) explains each step.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from gridfm_trustlayer import (
    ChannelSet,
    KernelIntervalCalibrator,
    NaiveIntervalCalibrator,
    OutageStrata,
    check_inputs,
    coverage_report,
    screening_report,
)
from gridfm_trustlayer.conformal.tuning import tune_n_eff_targets

DATA = Path("../demo/data")
if not DATA.is_dir():
    raise FileNotFoundError(
        "Run this notebook from examples/notebooks of the repository."
    )


def wide(frame, column, element):
    """One row per scenario and one column per element."""
    return frame.pivot(index="scenario", columns=element, values=column).sort_index()


data = {
    split: tuple(
        pd.read_parquet(DATA / split / "raw" / f"{table}_data.parquet")
        for table in ("bus", "branch", "gen")
    )
    for split in ("calibration", "test", "new")
}
for split, (bus, branch, gen) in data.items():
    print(
        f"{split}: {bus['scenario'].nunique():,} scenarios, columns of the bus table: {list(bus.columns)}"
    )

calibration: 9,723 scenarios, columns of the bus table: ['scenario', 'load_scenario_idx', 'bus', 'Pd', 'Qd', 'Pg', 'Qg', 'Vm', 'Va', 'PQ', 'PV', 'REF', 'vn_kv']
test: 4,848 scenarios, columns of the bus table: ['scenario', 'load_scenario_idx', 'bus', 'Pd', 'Qd', 'Pg', 'Qg', 'Vm', 'Va', 'PQ', 'PV', 'REF', 'vn_kv']
new: 294 scenarios, columns of the bus table: ['scenario', 'load_scenario_idx', 'bus', 'Pd', 'Qd', 'PQ', 'PV', 'REF', 'vn_kv']


## 1. A model of your own

The model sees the loads of each scenario and which lines and generators are out of service. It is a linear regression on the loads and on every pair of outages, fitted on half of the calibration scenarios; the other half calibrates the bounds. Any other model works the same way: trustlayer only sees its predictions.

The true values come from the AC power flow solution in the data: `Vm` per bus, and the loading of each line computed from the voltages and the line admittances, `max(|S_from|, |S_to|) / rate_a`. With your own data, they would come from your own solver.

In [2]:
def model_inputs(split):
    """What the model sees: loads (in per unit) and in-service flags, per scenario."""
    bus, branch, gen = data[split]
    loads = pd.concat([wide(bus, "Pd", "bus"), wide(bus, "Qd", "bus")], axis=1) / 100
    lines_in = wide(branch, "br_status", "idx").astype(bool)
    generators_in = wide(gen, "in_service", "idx").astype(bool)
    return loads, lines_in, generators_in


def features(loads, lines_in, generators_in):
    """Loads and every pair of outages (an outage paired with itself is the outage)."""
    out = 1.0 - np.hstack([lines_in, generators_in])
    first, second = np.triu_indices(out.shape[1])
    return np.hstack([np.ones((len(loads), 1)), loads, out[:, first] * out[:, second]])


def true_values(split):
    """Vm per bus and the loading of each line, from the AC power flow solution."""
    bus, branch, _ = data[split]
    vm = wide(bus, "Vm", "bus")
    voltage = vm.to_numpy() * np.exp(1j * np.radians(wide(bus, "Va", "bus").to_numpy()))
    lines = branch[branch["scenario"] == branch["scenario"].min()].sort_values("idx")
    v_from = voltage[:, lines["from_bus"].to_numpy()]
    v_to = voltage[:, lines["to_bus"].to_numpy()]
    y = {
        name: (
            wide(branch, f"{name}_r", "idx") + 1j * wide(branch, f"{name}_i", "idx")
        ).to_numpy()
        for name in ("Yff", "Yft", "Ytf", "Ytt")
    }
    s_from = np.abs(v_from * np.conj(y["Yff"] * v_from + y["Yft"] * v_to)) * 100  # MVA
    s_to = np.abs(v_to * np.conj(y["Ytt"] * v_to + y["Ytf"] * v_from)) * 100
    loading = np.maximum(s_from, s_to) / lines["rate_a"].to_numpy()
    return vm, pd.DataFrame(loading, index=vm.index, columns=lines["idx"].to_numpy())


inputs = {split: model_inputs(split) for split in data}
calibration_vm, calibration_loading = true_values("calibration")
scenarios = calibration_vm.index.to_numpy()
train = scenarios < np.median(scenarios)  # the model's half; the rest calibrates
x = features(*inputs["calibration"])[train]
y = np.hstack([calibration_vm, calibration_loading])[train]
weights = np.linalg.solve(x.T @ x + 0.1 * np.eye(x.shape[1]), x.T @ y)


def predict(split):
    """The model's Vm and line loading, one row per scenario."""
    output = features(*inputs[split]) @ weights
    index = inputs[split][0].index
    n_buses = calibration_vm.shape[1]
    return (
        pd.DataFrame(output[:, :n_buses], index=index, columns=calibration_vm.columns),
        pd.DataFrame(
            output[:, n_buses:], index=index, columns=calibration_loading.columns
        ),
    )


test_vm, test_loading = true_values("test")
vm_pred, loading_pred = predict("test")
in_service = inputs["test"][1].to_numpy()
print(
    f"Test scenarios: mean absolute error {np.abs(vm_pred - test_vm).to_numpy().mean():.4f} p.u. on Vm, "
    f"{np.abs(loading_pred - test_loading).to_numpy()[in_service].mean():.3f} on the loading of lines in service"
)

Test scenarios: mean absolute error 0.0052 p.u. on Vm, 0.024 on the loading of lines in service


## 2. Long tables

Trustlayer takes one row per scenario and element, with the prediction, the true value when it is known, and whether the value should get bounds. Two values should not:

- `Vm` at generator buses is a set-point, an input of the power flow rather than something to predict: only buses of type PQ (the `PQ` column) are calibrated;
- lines out of service carry no flow (`br_status` is 0).

In [3]:
def long(frame, element, name):
    return frame.rename_axis(index="scenario", columns=element).stack().rename(name)


def tables(split, scenarios=None):
    """One row per scenario and bus, and one per scenario and line."""
    bus, branch, _ = data[split]
    vm, loading = predict(split)
    buses = bus.set_index(["scenario", "bus"])[["PQ"]].join(long(vm, "bus", "Vm_pred"))
    lines = branch.rename(columns={"idx": "line"}).set_index(["scenario", "line"])[
        ["br_status"]
    ]
    lines = lines.join(long(loading, "line", "loading_pred"))
    if split != "new":  # new scenarios have no AC solution
        vm, loading = true_values(split)
        buses = buses.join(long(vm, "bus", "Vm_true"))
        lines = lines.join(long(loading, "line", "loading_true"))
    buses, lines = buses.reset_index(), lines.reset_index()
    if scenarios is not None:
        buses = buses[buses["scenario"].isin(scenarios)]
        lines = lines[lines["scenario"].isin(scenarios)]
    return buses, lines


calibration_buses, calibration_lines = tables("calibration", scenarios[~train])
calibration_lines.head()

,scenario,line,br_status,loading_pred,loading_true
97220,4861,0,0,0.018739,0.000000
97221,4861,1,1,1.929625,2.619431
97222,4861,2,1,0.446897,0.449946
97223,4861,3,1,0.425488,0.356488
97224,4861,4,1,0.993105,1.301834


## 3. Strata

Errors grow with the number of outages, so the bounds are fitted per number of outages. `OutageStrata` counts them from the in-service flags as the command-line tool does: it learns from the calibration set which lines and generators exist (in service in at least one scenario), then counts those out of service in any dataset.

In [4]:
_, lines_in, generators_in = inputs["calibration"]
outages = OutageStrata.fit(lines_in, generators_in)
n_outages = {split: outages.n_outages(*inputs[split][1:]) for split in data}
n_outages["calibration"].loc[scenarios[~train]].value_counts().sort_index().rename(
    "calibration scenarios"
)

n_outages
1    1255
2    1111
3    1019
4     853
5     624
Name: calibration scenarios, dtype: int64

## 4. ChannelSets and checks

`ChannelSet.from_frame` builds the predictions and the true values from the same rows, one set for the buses and one for the lines. The kernel method also needs one vector per scenario that describes it. This model has no embeddings, but any description of the scenario can be tried. Its predictions (`Vm` at every bus and the loading of every line) are used here; the raw inputs (loads and in-service flags) were tried as well and did not help (see the end of section 5). Whichever you use, check it the same way.

In [5]:
def scenario_vectors(split):
    """One vector per scenario for the kernel method: the model's predictions."""
    vm, loading = predict(split)
    vectors = pd.concat([vm, loading], axis=1)
    vectors.columns = [f"z{i}" for i in range(vectors.shape[1])]
    return vectors


def channel_sets(split, buses, lines, column):
    """Bus and line ChannelSets of one column (`pred` or `true`)."""
    common = {
        "strata": {"n_outages": n_outages[split]},
        "embeddings": scenario_vectors(split),
    }
    return [
        ChannelSet.from_frame(
            buses, {"Vm": f"Vm_{column}"}, entity="bus", valid="PQ", **common
        ),
        ChannelSet.from_frame(
            lines,
            {"line_loading": f"loading_{column}"},
            entity="line",
            entity_name="branch",
            valid="br_status",
            **common,
        ),
    ]


calibration_pred = channel_sets(
    "calibration", calibration_buses, calibration_lines, "pred"
)
calibration_true = channel_sets(
    "calibration", calibration_buses, calibration_lines, "true"
)
for predictions, truth in zip(calibration_pred, calibration_true):
    print(check_inputs(predictions, truth, strat="n_outages", alpha=0.1, kernel=True))

No problems found.


The warning is expected: bus 0 is the reference bus, never of type PQ, so its voltage gets no bounds, as intended. `check_inputs` reports errors to fix (misaligned scenarios, missing embeddings, non-integer strata, ...) and warnings to read; see the guide's section on what it reports.

## 5. Fit the bounds

Both methods give 90% intervals (`alpha=0.1`, 5% per tail), fitted per number of outages. For the kernel method, `tune_n_eff_targets` chooses how many calibration scenarios weigh in each bound (`n_eff_target`) by cross-validation on the calibration set alone, for the buses and the lines together.

In [6]:
settings = {"alpha": 0.1, "strat": "n_outages"}
scp = [
    NaiveIntervalCalibrator.fit(
        predictions, truth, channel_names=tuple(predictions.channels), **settings
    )
    for predictions, truth in zip(calibration_pred, calibration_true)
]
tuning = tune_n_eff_targets(
    {
        name: (predictions, truth, tuple(predictions.channels))
        for name, predictions, truth in zip(
            ("buses", "lines"), calibration_pred, calibration_true
        )
    },
    **settings,
)
kcp = [
    KernelIntervalCalibrator.fit(
        predictions,
        truth,
        channel_names=tuple(predictions.channels),
        embeddings=predictions.scenario_embeddings,
        n_eff_target=result.selected_n_eff_target,
        **settings,
    )
    for predictions, truth, result in zip(
        calibration_pred, calibration_true, tuning.values()
    )
]
for name, result in tuning.items():
    print(
        f"{name}: n_eff_target = {result.selected_n_eff_target:.0f} ({result.reason})"
    )

buses: n_eff_target = 600 (narrowest eligible candidate)
lines: n_eff_target = 600 (narrowest eligible candidate)


## 6. Coverage on test scenarios

The test set was generated separately, with its own AC solutions. `coverage_report` gives the share of true values inside their bounds, overall and per number of outages, and the mean width of the bounds.

In [7]:
def intervals(calibrators, sets):
    """`{variable: (lower, upper)}` of every variable of the sets."""
    bounds = {}
    for calibrator, predictions in zip(calibrators, sets):
        bounds.update(calibrator.predict_intervals(predictions))
    return bounds


test_buses, test_lines = tables("test")
test_pred = channel_sets("test", test_buses, test_lines, "pred")
test_true = channel_sets("test", test_buses, test_lines, "true")
test_bounds = {"SCP": intervals(scp, test_pred), "KCP": intervals(kcp, test_pred)}
coverage = {
    method: coverage_report(test_pred, test_true, bounds, alpha=0.1, strat="n_outages")
    for method, bounds in test_bounds.items()
}
comparison = pd.concat(
    {
        method: table.set_index(["variable", "stratum"])[["coverage", "mean_width"]]
        for method, table in coverage.items()
    },
    axis=1,
)
comparison.round(3)

SCP                 KCP           
                         coverage mean_width coverage mean_width
variable     stratum                                            
Vm           all            0.907      0.030    0.911      0.027
             n_outages=1    0.899      0.013    0.922      0.012
             n_outages=2    0.912      0.020    0.911      0.018
             n_outages=3    0.904      0.028    0.897      0.025
             n_outages=4    0.911      0.042    0.916      0.039
             n_outages=5    0.909      0.064    0.909      0.061
line_loading all            0.904      0.104    0.905      0.100
             n_outages=1    0.900      0.064    0.902      0.062
             n_outages=2    0.900      0.079    0.902      0.077
             n_outages=3    0.902      0.111    0.898      0.102
             n_outages=4    0.911      0.141    0.915      0.135
             n_outages=5    0.915      0.191    0.919      0.190

In [8]:
for variable in ("Vm", "line_loading"):
    scp_row, kcp_row = (
        coverage[method].set_index(["variable", "stratum"]).loc[(variable, "all")]
        for method in ("SCP", "KCP")
    )
    print(
        f"{variable}: SCP covers {scp_row.coverage:.1%} with a mean width of {scp_row.mean_width:.4f}; "
        f"KCP {kcp_row.coverage:.1%} with {kcp_row.mean_width:.4f} "
        f"({1 - kcp_row.mean_width / scp_row.mean_width:.0%} narrower)."
    )

Vm: SCP covers 90.7% with a mean width of 0.0296; KCP 91.1% with 0.0273 (8% narrower).
line_loading: SCP covers 90.4% with a mean width of 0.1037; KCP 90.5% with 0.0997 (4% narrower).


SCP covers 90% overall and within each number of outages, as guaranteed when the test scenarios are like the calibration ones. The bounds widen with the number of outages, where the model is less accurate. KCP stays close to 90% with narrower bounds: weighing the calibration scenarios by how close the model's predictions are, its margins follow the scenarios where the model does well or badly. Its coverage is checked here on test data, not guaranteed.

The raw inputs (loads and in-service flags) did not work as the scenario vector: in that many dimensions the calibration scenarios are all far from a new one, so with a small `n_eff_target` KCP falls back to the SCP bounds, and the cross-validation picks nothing better than SCP. Try the vectors you have and keep the one that checks out.

## 7. Screening against operating limits

With limits on `Vm` (0.95 to 1.05 p.u.) and on line loading (100% of the rating), every value is within limits, uncertain (its bounds cross a limit) or a violation, and a scenario takes the worst status of its buses and lines: secure, uncertain or insecure. Uncertain and insecure scenarios go to an AC power flow. With true values, `screening_report` also counts the violations caught and missed.

In [9]:
limits = {"Vm": {"lower": 0.95, "upper": 1.05}, "line_loading": {"upper": 1.0}}
screening = screening_report(
    test_pred, test_bounds["SCP"], limits, true_values=test_true, strat="n_outages"
)
alarms = screening.alarms
alarms[alarms["stratum"] == "all"].set_index(["variable", "method"])[
    ["violations", "alarms", "caught", "missed", "false_alarms"]
]

violations  alarms  caught  missed  false_alarms
variable     method                                                    
Vm           interval        4691   19002    4497     194         14505
             point           4691    5493    3424    1267          2069
line_loading interval         806    1263     774      32           489
             point            806     772     733      73            39

In [10]:
statuses = screening.statuses
scenario_counts = statuses[
    (statuses["level"] == "scenario")
    & (statuses["variable"] == "all")
    & (statuses["stratum"] == "all")
].set_index(["method", "status"])
n_scenarios = len(test_pred[0].scenario_ids)
for method, label in (
    ("point", "The point predictions alone"),
    ("interval", "The SCP bounds"),
):
    secure = scenario_counts.loc[(method, "secure")]
    print(
        f"{label} clear {secure.n:,} of {n_scenarios:,} scenarios as secure; "
        f"{secure.truly_violated:,} of them ({secure.truly_violated / secure.n:.0%}) are truly insecure."
    )

The point predictions alone clear 2,130 of 4,848 scenarios as secure; 313 of them (15%) are truly insecure.
The SCP bounds clear 320 of 4,848 scenarios as secure; 17 of them (5%) are truly insecure.


The model's point predictions miss many violations: a scenario they call secure is often insecure. The bounds rarely let a violation through, but this model is not accurate enough to clear most scenarios. The bounds say how much of the screening the model can take on; a better model clears more of them.

## 8. New scenarios

For new scenarios only the model's predictions exist. The same steps give their bounds and status: tables without the true values, `ChannelSet`s with the same strata and scenario vectors, then `predict_intervals`. `to_frame` returns the bounds and status of every value as a long table, to join back onto your own data.

In [11]:
new_buses, new_lines = tables("new")
new_pred = channel_sets("new", new_buses, new_lines, "pred")
for predictions, calibrator in zip(new_pred, scp):
    print(check_inputs(predictions, calibrator=calibrator))
new_bounds = intervals(scp, new_pred)
lines_out = new_pred[1].to_frame(new_bounds, entity="line", limits=limits)
lines_out[
    lines_out["line_loading_valid"]
    & (lines_out["line_loading_status"] != "within_limits")
].head()

No problems found.
No problems found.


,scenario,line,n_outages,line_loading,line_loading_valid,line_loading_lower,line_loading_upper,line_loading_status,line_loading_point_status
21,1,1,2,1.892865,True,1.752017,2.018261,violation,violation
101,5,1,5,2.273371,True,1.950102,2.737107,violation,violation
221,11,1,4,2.195306,True,1.964178,2.512108,violation,violation
461,23,1,2,1.904990,True,1.764142,2.030386,violation,violation
781,39,1,5,0.932172,True,0.608903,1.395908,uncertain,within_limits


In [12]:
report = screening_report(new_pred, new_bounds, limits, strat="n_outages")
report.scenarios["status"].value_counts().rename("new scenarios")

status
uncertain    180
insecure      70
secure        44
Name: new scenarios, dtype: int64

## Next steps

- Save a fitted calibrator with `save_interval_artifact` and load it with `load_interval_artifact`.
- [Using trustlayer with your own data](https://gridfm.github.io/gridfm-trustlayer/user-guide/own-data/): the inputs, the mistakes to avoid and what `check_inputs` reports.
- [The kernel method](https://gridfm.github.io/gridfm-trustlayer/user-guide/kernel/) and the [API reference](https://gridfm.github.io/gridfm-trustlayer/reference/api/).